
## Step 0 — Environment

This notebook was created and run locally in VS Code using the course `uv`
Python 3.12 environment. The course folder contains the `.venv` environment,
`pyproject.toml`, and `uv.lock` files, confirming that the required local
environment and package setup are available.

In [1]:
import pandas, statsmodels, yfinance, sys

print(
    sys.version.split()[0],
    pandas.__version__,
    statsmodels.__version__,
    yfinance.__version__
)

3.12.14 2.2.3 0.15.0 1.7.0


## Step 1 — Conventions

Return type: simple monthly returns from adjusted month-end close downloaded
with `yfinance` using `auto_adjust=True`.

Frequency: monthly; annualization will use ×12 for mean returns and alphas,
and ×√12 for volatility.

Risk-free: monthly `RF` from the Ken French Data Library’s Fama/French
3 Factors file.

Window: 2015-07 to 2025-06, 120 months.

Market series: Fama/French `Mkt-RF` for factor regressions and `SPY` as the
investable market benchmark.

Equal-weight: rebalanced monthly across 11 securities, with a weight of 1/11
in each security.

Standard deviation: sample standard deviation (`ddof=1`).

Panel: BAC, C, CFG, IBM, JPM, KEY, PNC, TFC, USB, WFC, and ZION.

## Step 2 — Build the Excess-Return Panel

I retain the bank-focused panel from Lab 2 but exclude Credit Suisse, First
Republic, Signature Bank, and Silicon Valley Bank because they do not provide
complete price histories through June 2025. CMA was replaced with CFG because
yfinance returned zero rows for CMA despite its CRSP coverage in Lab 2.

In [2]:
import pandas as pd
import yfinance as yf
from pathlib import Path

SECURITIES = [
    "BAC", "C", "CFG", "IBM", "JPM", "KEY",
    "PNC", "TFC", "USB", "WFC", "ZION"
]

TICKERS = SECURITIES + ["SPY"]

WINDOW_START = "2015-07"
WINDOW_END = "2025-06"
EXPECTED_MONTHS = 120

# Start one month early to calculate the July 2015 return
PRICE_START = "2015-06-01"
PRICE_END = "2025-07-01"

ASSET_COLUMNS = TICKERS

In [3]:
# This works whether the notebook is running from the main folder or lab3
ff_file = Path("lab3/F-F_Research_Data_Factors.csv")

if not ff_file.exists():
    ff_file = Path("F-F_Research_Data_Factors.csv")

assert ff_file.exists(), "Put F-F_Research_Data_Factors.csv in the lab3 folder."

lines = ff_file.read_text(encoding="latin1").splitlines()

# 1. Parses the monthly block only
header_row = next(
    i for i, line in enumerate(lines)
    if line.strip().startswith(",Mkt-RF")
)

end_row = next(
    i for i in range(header_row + 1, len(lines))
    if not lines[i].strip()
)

ff = pd.read_csv(
    ff_file,
    skiprows=header_row,
    nrows=end_row - header_row - 1
)

ff.columns = ff.columns.str.strip()
ff = ff.rename(columns={ff.columns[0]: "Month"})

# 3. Keys the rows on a monthly period
ff["Month"] = pd.to_datetime(
    ff["Month"].astype(str),
    format="%Y%m"
).dt.to_period("M")

# 2. Divides percentage returns by 100
ff = (
    ff.set_index("Month")[["Mkt-RF", "SMB", "HML", "RF"]]
    .apply(pd.to_numeric, errors="coerce")
    .div(100)
    .loc[WINDOW_START:WINDOW_END]
)

ff.head()

,Mkt-RF,SMB,HML,RF
Month,,,,
2015-07,0.0157,-0.0417,-0.0416,0.0
2015-08,-0.0602,0.0037,0.0275,0.0
2015-09,-0.0306,-0.0263,0.0055,0.0
2015-10,0.0775,-0.0191,-0.0046,0.0
2015-11,0.0057,0.0358,-0.0043,0.0


In [4]:
# Confirm the expected Fama/French columns were parsed
print(ff.columns.tolist())

['Mkt-RF', 'SMB', 'HML', 'RF']


In [5]:
prices = yf.download(
    TICKERS,
    start=PRICE_START,
    end=PRICE_END,
    auto_adjust=True,
    progress=False
)["Close"]

monthly_prices = prices.resample("ME").last()

raw_monthly_returns = monthly_prices.pct_change(fill_method=None)
raw_monthly_returns.index = raw_monthly_returns.index.to_period("M")
raw_monthly_returns = raw_monthly_returns.loc[WINDOW_START:WINDOW_END]

raw_monthly_returns.head()

Ticker,BAC,C,CFG,IBM,JPM,KEY,PNC,SPY,TFC,USB,WFC,ZION
Date,,,,,,,,,,,,
2015-07,0.050529,0.059190,-0.041719,-0.004119,0.017974,-0.011984,0.031915,0.022589,-0.000992,0.041705,0.028983,-0.017328
2015-08,-0.086130,-0.085186,-0.047948,-0.079463,-0.064644,-0.069094,-0.071908,-0.060950,-0.076991,-0.063260,-0.072436,-0.068375
2015-09,-0.043442,-0.072364,-0.038678,-0.019744,-0.048830,-0.053129,-0.021071,-0.025516,-0.035753,-0.025679,-0.037127,-0.050345
2015-10,0.077022,0.072740,0.018441,-0.033731,0.061401,-0.045350,0.017669,0.085060,0.043540,0.028529,0.054333,0.044662
2015-11,0.038737,0.017303,0.100385,0.004624,0.037821,0.061610,0.058165,0.003655,0.046883,0.040541,0.024727,0.043486


In [6]:
# Join raw monthly security returns to the monthly Fama/French factor data
# An inner join keeps only months available in both sources
joined = raw_monthly_returns.join(ff, how="inner")

# Define the 120 monthly periods declared in Step 1
expected_index = pd.period_range(WINDOW_START, WINDOW_END, freq="M")

# Confirm that no months were lost during the join
assert joined.index.equals(expected_index), (
    f"Expected {WINDOW_START} to {WINDOW_END}, but the joined data runs "
    f"{joined.index.min()} to {joined.index.max()} with {len(joined)} months."
)

# Check that each security and SPY has a valid return in every joined month
missing_by_column = joined[ASSET_COLUMNS].isna().sum()

assert (missing_by_column == 0).all(), (
    "Missing monthly returns:\n"
    + missing_by_column[missing_by_column > 0].to_string()
)

# Confirm the observed month count matches the Step 1 convention
assert len(joined) == EXPECTED_MONTHS

print(f"joined: {joined.shape}    {joined.index.min()} -> {joined.index.max()}")

# Preserve the raw joined frame for later wealth and CAGR calculations
# Create a separate frame for excess-return analysis
excess_returns = joined.copy()

# Subtract RF from stocks and SPY only
excess_returns[ASSET_COLUMNS] = joined[ASSET_COLUMNS].sub(
    joined["RF"],
    axis=0
)

# Do not subtract RF from Mkt-RF
show_columns = ["JPM", "SPY", "Mkt-RF", "RF"]


print("First row:")
display(joined.loc[[joined.index[0]], show_columns].round(4))

print("Last row:")
display(joined.loc[[joined.index[-1]], show_columns].round(4))

joined: (120, 16)    2015-07 -> 2025-06
First row:


,JPM,SPY,Mkt-RF,RF
Date,,,,
2015-07,0.018,0.0226,0.0157,0.0


Last row:


,JPM,SPY,Mkt-RF,RF
Date,,,,
2025-06,0.0981,0.0514,0.0486,0.0034


## Open data-quality issue — survivorship and panel coverage

The 120-month return panel is a continuous-coverage, investable subset rather
than the original 15-security Lab 2 bank-stress universe. Credit Suisse, First
Republic, Signature Bank, and Silicon Valley Bank were excluded because they
did not remain available through the June 2025 endpoint. CMA also returned zero
rows from yfinance, so CFG was substituted.

This construction can bias the equal-weight panel’s later alpha upward because
the omitted firms include distressed institutions that exited the sample. The
panel can therefore support analysis of the surviving/continuous-coverage
securities, but not a headline performance or alpha claim about the complete
original Lab 2 universe without a delisting-return or CRSP-based treatment.

## Checkpoint 4 — Stopping point

I completed Steps 0–2: configured the local VS Code environment, stated the
return conventions, downloaded and parsed the monthly Fama/French factors, and
built and validated the 120-month raw and excess-return panel. I stopped after
confirming the joined panel runs from 2015-07 through 2025-06 with no missing
monthly returns.

## AI Disclosure

ChatGPT (OpenAI) was used to help set up VS Code and the local Python
environment, explain the assignment instructions, troubleshoot file-path and
`yfinance` errors, and draft/review Python code for parsing the Fama/French
file, calculating monthly returns, and validating the joined panel. I selected
the bank-focused panel, ran the code locally, reviewed the outputs, and changed
CMA to CFG after `yfinance` returned no CMA data.

Codex CLI was used to confirm that it could access the course folder by listing
its files. It was not used to run the Lab 3 analysis.

In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

MONTHS_PER_YEAR = 12

# All stock columns, excluding the benchmark and Fama-French factor columns
exclude_cols = ["SPY", "Mkt-RF", "SMB", "HML", "RF"]
security_cols = [col for col in joined.columns if col not in exclude_cols]

print("Securities used in the equal-weight panel:")
print(security_cols)
print("Number of securities:", len(security_cols))

asset_returns = joined[security_cols].copy()
asset_excess = excess_returns[security_cols].copy()
mkt_excess = joined["Mkt-RF"].copy()

ew_return = asset_returns.mean(axis=1).rename("EW panel")
ew_excess = asset_excess.mean(axis=1).rename("EW panel")

returns_all = pd.concat([asset_returns, ew_return], axis=1)
excess_all = pd.concat([asset_excess, ew_excess], axis=1)

assert returns_all.isna().sum().sum() == 0
assert excess_all.isna().sum().sum() == 0

print(returns_all.shape, returns_all.index.min(), "to", returns_all.index.max())

Securities used in the equal-weight panel:
['BAC', 'C', 'CFG', 'IBM', 'JPM', 'KEY', 'PNC', 'TFC', 'USB', 'WFC', 'ZION']
Number of securities: 11
(120, 12) 2015-07 to 2025-06


In [8]:
def cagr(monthly_returns):
    n_months = len(monthly_returns)
    return (1 + monthly_returns).prod() ** (MONTHS_PER_YEAR / n_months) - 1

step3 = pd.DataFrame(index=returns_all.columns)

# CAGR uses ordinary total returns
step3["CAGR"] = returns_all.apply(cagr)

# Volatility is based on monthly excess returns, annualized
step3["ann vol"] = excess_all.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)

# Correct Week 3 Sharpe: annualized arithmetic mean excess return / annualized volatility
step3["Sharpe (excess, arithmetic)"] = (
    excess_all.mean() * MONTHS_PER_YEAR
) / step3["ann vol"]

# Shortcut for comparison only
step3["CAGR/vol, rf=0"] = step3["CAGR"] / step3["ann vol"]

step3_display = step3.copy()
for col in ["CAGR", "ann vol"]:
    step3_display[col] = step3_display[col].map(lambda x: f"{x:.2%}")
for col in ["Sharpe (excess, arithmetic)", "CAGR/vol, rf=0"]:
    step3_display[col] = step3_display[col].map(lambda x: f"{x:.3f}")

step3_display

,CAGR,ann vol,"Sharpe (excess, arithmetic)","CAGR/vol, rf=0"
BAC,13.17%,29.75%,0.502,0.443
C,7.39%,32.59%,0.326,0.227
CFG,8.91%,36.45%,0.372,0.245
IBM,11.29%,23.97%,0.489,0.471
JPM,18.86%,24.50%,0.755,0.770
KEY,5.63%,34.50%,0.281,0.163
PNC,10.33%,27.02%,0.430,0.382
TFC,4.66%,28.38%,0.241,0.164
USB,4.03%,26.56%,0.212,0.152
WFC,6.59%,29.33%,0.302,0.225


### Step 3 Interpretation

Rank changes occur between BAC and IBM; among C, WFC, and ZION; and between KEY and TFC when comparing the two measures. I will use the Sharpe ratio based on arithmetic excess returns for the rest of the course because it is the Week 3 standard and properly accounts for the risk-free rate.

In [9]:
X = sm.add_constant(mkt_excess)

jensen_rows = []
models = {}

for name in excess_all.columns:
    model = sm.OLS(excess_all[name], X).fit()
    models[name] = model
    
    jensen_rows.append({
        "security": name,
        "α/yr": model.params["const"] * MONTHS_PER_YEAR,
        "t(α)": model.tvalues["const"],
        "β": model.params["Mkt-RF"],
        "t(β)": model.tvalues["Mkt-RF"],
        "R²": model.rsquared,
        "resid vol": np.sqrt(model.mse_resid * MONTHS_PER_YEAR)
    })

step4 = pd.DataFrame(jensen_rows).set_index("security")

step4_display = step4.copy()
for col in ["α/yr", "resid vol"]:
    step4_display[col] = step4_display[col].map(lambda x: f"{x:.2%}")
for col in ["t(α)", "β", "t(β)", "R²"]:
    step4_display[col] = step4_display[col].map(lambda x: f"{x:.3f}")

step4_display

,α/yr,t(α),β,t(β),R²,resid vol
security,,,,,,
BAC,-1.97%,-0.317,1.415,12.920,0.586,19.22%
C,-7.70%,-1.112,1.532,12.571,0.573,21.40%
CFG,-3.91%,-0.432,1.461,9.169,0.416,27.97%
IBM,1.10%,0.177,0.889,8.080,0.356,19.31%
JPM,5.04%,0.942,1.126,11.936,0.547,16.56%
KEY,-6.09%,-0.690,1.322,8.506,0.380,27.28%
PNC,-2.38%,-0.380,1.173,10.601,0.488,19.42%
TFC,-6.70%,-0.950,1.134,9.125,0.414,21.82%
USB,-7.02%,-1.062,1.060,9.098,0.412,20.45%


### Step 4 Interpretation

I would not treat any alpha as statistically reliable because none has an absolute alpha t-statistic of at least about 1.98, the usual 5% significance threshold. This does not mean the banks performed poorly: several had positive returns and positive Sharpe ratios, but their returns were not reliably higher than CAPM would predict given their market risk. The EW panel’s alpha t-statistic is also not significant because, although diversification lowered residual volatility, its estimated alpha was still small relative to the uncertainty around that estimate.

In [10]:
market_ann_vol = mkt_excess.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)

step5 = pd.DataFrame(index=excess_all.columns)
step5["β"] = step4["β"]

# This must match the covariance-matrix diagonal later
step5["total"] = excess_all.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)

step5["systematic"] = step5["β"].abs() * market_ann_vol
step5["idiosyncratic"] = step4["resid vol"]

# Variance share explained by systematic market risk
step5["share"] = (step5["systematic"] ** 2) / (step5["total"] ** 2)

step5_display = step5.copy()
step5_display["β"] = step5_display["β"].map(lambda x: f"{x:.3f}")
for col in ["total", "systematic", "idiosyncratic", "share"]:
    step5_display[col] = step5_display[col].map(lambda x: f"{x:.1%}")

step5_display

,β,total,systematic,idiosyncratic,share
BAC,1.415,29.7%,22.8%,19.2%,58.6%
C,1.532,32.6%,24.7%,21.4%,57.3%
CFG,1.461,36.5%,23.5%,28.0%,41.6%
IBM,0.889,24.0%,14.3%,19.3%,35.6%
JPM,1.126,24.5%,18.1%,16.6%,54.7%
KEY,1.322,34.5%,21.3%,27.3%,38.0%
PNC,1.173,27.0%,18.9%,19.4%,48.8%
TFC,1.134,28.4%,18.3%,21.8%,41.4%
USB,1.060,26.6%,17.1%,20.4%,41.2%
WFC,1.156,29.3%,18.6%,22.8%,40.2%


In [11]:
check_name = "AAPL" if "AAPL" in step5.index else security_cols[0]

reconstructed_total = np.sqrt(
    step5.loc[check_name, "systematic"] ** 2
    + step5.loc[check_name, "idiosyncratic"] ** 2
)

print(
    f"check {check_name}: sqrt("
    f"{step5.loc[check_name, 'systematic']:.2%}² + "
    f"{step5.loc[check_name, 'idiosyncratic']:.2%}²) = "
    f"{reconstructed_total:.2%}"
)
print(f"reported total volatility = {step5.loc[check_name, 'total']:.2%}")

check BAC: sqrt(22.77%² + 19.22%²) = 29.80%
reported total volatility = 29.75%


In [12]:
median_single_idio = step5.loc[security_cols, "idiosyncratic"].median()
ew_idio = step5.loc["EW panel", "idiosyncratic"]

print(f"EW panel idiosyncratic volatility: {ew_idio:.2%}")
print(f"Median single-name idiosyncratic volatility: {median_single_idio:.2%}")

EW panel idiosyncratic volatility: 18.07%
Median single-name idiosyncratic volatility: 21.40%


# Step 5

The equal-weight panel’s idiosyncratic volatility is 18.07% compared with a median single-name idiosyncratic volatility of 21.40%. This shows that equal-weighting across 11 securities materially reduced company-specific risk, although it did not remove market-wide systematic risk as all the securities were banks that share similar market exposure

In [13]:
# Keep original security order from Part 1
week4_excess_returns = asset_excess[security_cols].copy()
week4_cov = week4_excess_returns.cov() * MONTHS_PER_YEAR

LAB3_OUTPUT_DIR = Path("lab3") if Path("lab3").exists() else Path(".")
returns_file = LAB3_OUTPUT_DIR / "week4_monthly_excess_returns.csv"
cov_file = LAB3_OUTPUT_DIR / "week4_annualized_excess_covariance.csv"

week4_excess_returns.to_csv(returns_file, index_label="Date")
week4_cov.to_csv(cov_file, index_label="Security")

# Reload and confirm the order is preserved
returns_check = pd.read_csv(returns_file, index_col="Date")
cov_check = pd.read_csv(cov_file, index_col="Security")

assert returns_check.columns.tolist() == security_cols
assert cov_check.index.tolist() == security_cols
assert cov_check.columns.tolist() == security_cols

print("Week 4 files saved and column order verified.")
print("\n3×3 covariance-matrix corner:")
display(week4_cov.iloc[:3, :3])

Week 4 files saved and column order verified.

3×3 covariance-matrix corner:


,BAC,C,CFG
BAC,0.088483,0.084526,0.095079
C,0.084526,0.106193,0.095944
CFG,0.095079,0.095944,0.132888


In [14]:
diag_check = pd.DataFrame({
    "covariance diagonal": np.diag(week4_cov),
    "Step 5 total vol squared": step5.loc[security_cols, "total"] ** 2
}, index=security_cols)

diag_check["difference"] = (
    diag_check["covariance diagonal"] - diag_check["Step 5 total vol squared"]
)

display(diag_check.head())

assert np.allclose(
    diag_check["covariance diagonal"],
    diag_check["Step 5 total vol squared"]
)

print("Diagonal check passed: covariance diagonal equals Step 5 total volatility squared.")

,covariance diagonal,Step 5 total vol squared,difference
BAC,0.088483,0.088483,1.387779e-17
C,0.106193,0.106193,1.387779e-17
CFG,0.132888,0.132888,-2.775558e-17
IBM,0.057432,0.057432,1.387779e-17
JPM,0.060048,0.060048,2.775558e-17


Diagonal check passed: covariance diagonal equals Step 5 total volatility squared.


**Stopping point:** Completed Steps 3–5, including the saved and verified Week 4 excess-return and covariance-matrix inputs.

**AI disclosure:** AI was used to help explain the required calculations and write Python code. I reviewed, ran, and verified all code and the interpretations myself. I made sure to review the output and ensure that the values made sense based on the given securities and were consistent throughout each step. 

# Exercise 3 — Lab 3 Part 3

The following cells complete Steps 6–9. They retain the Lab 2 catalog's open Blocker rather than treating the survivor-only panel as a full bank-sector sample.

## Exercise 3 — Step 6: Independent beta verification

I verify **JPM** against the Yahoo Finance quote page, which reports **Beta (5Y Monthly) = 0.98**, retrieved 2026-09-20. Yahoo does not state all of its calculation conventions, so the comparison below is explicitly provisional.

In [15]:
# Step 6: matched, independently sourced five-year monthly beta check for JPM.
# The 60 complete months September 2021--August 2026 require an August 2021 price.
import yfinance as yf
VERIFY_TICKER = "JPM"
VENDOR_BETA = 0.98
VENDOR_RETRIEVAL_DATE = "2026-09-20"
VERIFY_PRICE_START = "2021-08-01"
VERIFY_PRICE_END = "2026-09-01"
verify_prices = yf.download([VERIFY_TICKER, "SPY"], start=VERIFY_PRICE_START,
                            end=VERIFY_PRICE_END, auto_adjust=True, progress=False)["Close"]
verify_returns = verify_prices.resample("ME").last().pct_change(fill_method=None).dropna()
verify_returns.index = verify_returns.index.to_period("M")
assert len(verify_returns) == 60
verify_model = sm.OLS(verify_returns[VERIFY_TICKER], sm.add_constant(verify_returns["SPY"])).fit()
matched_beta = verify_model.params["SPY"]
step6_beta = pd.DataFrame({
    "beta": [VENDOR_BETA, matched_beta],
    "window": ["Yahoo does not state; labelled 5Y Monthly", f"{verify_returns.index.min()} through {verify_returns.index.max()} (60 complete months)"],
    "conventions": ["Not fully disclosed by Yahoo", "Simple returns from adjusted month-end closes; OLS with intercept on SPY"],
}, index=["Yahoo Finance Beta (5Y Monthly)", "Matched rerun (JPM on SPY)"])
step6_beta.to_csv(LAB3_OUTPUT_DIR / "step6_jpm_beta_verification.csv", index_label="method")
display(step6_beta)
print(f"Yahoo: {VENDOR_BETA:.2f}; matched rerun: {matched_beta:.3f}; absolute gap: {abs(matched_beta - VENDOR_BETA):.3f}.")

,beta,window,conventions
Yahoo Finance Beta (5Y Monthly),0.980000,Yahoo does not state; labelled 5Y Monthly,Not fully disclosed by Yahoo
Matched rerun (JPM on SPY),0.984671,2021-09 through 2026-08 (60 complete months),Simple returns from adjusted month-end closes;...


Yahoo: 0.98; matched rerun: 0.985; absolute gap: 0.005.


### Step 6 interpretation

My original Step 4 JPM beta was estimated on French Mkt-RF for 2015-07 through 2025-06, so it is not directly comparable to Yahoo's rolling five-year figure. I worked the checks in order: the identity is JPM; the frequency is monthly; the matched window is 2021-09 through 2026-08, using only 60 complete months; the return type is simple adjusted-close return; the regressor is SPY as the investable S&P 500 proxy; and the formula is OLS with an intercept. The rerun beta is reported below beside Yahoo's 0.98. Yahoo does not disclose whether it uses adjusted prices, its precise month-end rule, or its benchmark implementation, so I assume adjusted monthly total returns against the S&P 500 proxy and call the 0.005 match provisional. The remaining difference is consistent with Yahoo's two-decimal display, not evidence of a computation error.

## Exercise 3 — Step 7: Benchmark defence

The equal-weight panel is regressed on French Mkt-RF, the declared investable benchmark SPY, and the obvious equal-weighted large-cap alternative RSP. Each alpha is an annualized intercept from the same 2015-07 through 2025-06 sample.

In [16]:
# Step 7: benchmark sensitivity of the equal-weight panel alpha.
benchmark_prices = yf.download(["SPY", "RSP"], start=PRICE_START, end=PRICE_END,
                               auto_adjust=True, progress=False)["Close"]
benchmark_raw = benchmark_prices.resample("ME").last().pct_change(fill_method=None)
benchmark_raw.index = benchmark_raw.index.to_period("M")
benchmark_raw = benchmark_raw.loc[WINDOW_START:WINDOW_END]
assert benchmark_raw.index.equals(joined.index) and benchmark_raw.notna().all().all()
benchmark_excess = benchmark_raw.sub(joined["RF"], axis=0)
benchmark_regressors = {"French Mkt-RF": joined["Mkt-RF"], "SPY": benchmark_excess["SPY"], "RSP": benchmark_excess["RSP"]}
benchmark_rows = []
for label, regressor in benchmark_regressors.items():
    fitted = sm.OLS(ew_excess, sm.add_constant(regressor.rename("benchmark"))).fit()
    benchmark_rows.append({"benchmark": label, "alpha_per_year": fitted.params["const"] * MONTHS_PER_YEAR,
                           "t_alpha": fitted.tvalues["const"], "beta": fitted.params["benchmark"], "R_squared": fitted.rsquared})
step7 = pd.DataFrame(benchmark_rows).set_index("benchmark")
step7.to_csv(LAB3_OUTPUT_DIR / "step7_ew_benchmark_alphas.csv")
display(step7.style.format({"alpha_per_year": "{:.2%}", "t_alpha": "{:.3f}", "beta": "{:.3f}", "R_squared": "{:.3f}"}))

,alpha_per_year,t_alpha,beta,R_squared
benchmark,,,,
French Mkt-RF,-3.42%,-0.585,1.234,0.549
SPY,-3.66%,-0.602,1.242,0.514
RSP,-0.95%,-0.190,1.290,0.664


### Step 7 benchmark defence

**Strongest objection:** SPY is capitalization-weighted and dominated by the largest U.S. companies, while this equal-weighted, bank-heavy panel gives each constituent the same starting weight. RSP is therefore a more like-for-like investable comparator: the estimated equal-weight-panel alpha changes from the SPY result to the RSP result shown in the table, a difference of several percentage points per year. That sensitivity supports the colleague's warning that benchmark choice matters.

**Response:** I concede that RSP is a strong alternative for an equal-weight portfolio and report it rather than presenting SPY as uniquely correct. I retain Mkt-RF as the primary factor-model benchmark because it is the stated Week 3 market factor, and SPY as the primary investable benchmark because it is the broad, liquid capitalization-weighted alternative declared in Step 1. No choice makes this survivor-only bank-oriented panel a general U.S. equity portfolio, so the defensible conclusion is benchmark-sensitive descriptive evidence, not a benchmark-independent alpha claim.

## Exercise 3 — Step 8: What these numbers can and cannot support

The table and conclusions below respond to every open issue in `data_quality_issues.md`.

### What these numbers can and cannot support

A **Blocker remains open**, so I do **not** report a headline alpha for the full 2022–2023 U.S. bank sector. The calculated alpha, Sharpe, beta, volatility, and optimizer inputs describe only the continuous-coverage 11-name survivor subset, under the stated 2015-07 to 2025-06 conventions.

| Open issue | What it does to alpha | Response |
|---|---|---|
| Requested names absent from the price response — **Blocker** | Omitting failed or unavailable requested names can bias the survivor-panel alpha upward or downward; for a stress-era bank question, the direction cannot be assumed, and the full-universe alpha is not identified. | **Restrict the claim:** report only survivor-subset descriptive statistics; do not call them a survivorship-free bank-sector result. |
| End-of-sample universe construction — **High** | Selecting names with knowledge of end-of-sample survival conditions alpha on continued coverage and can make historical performance look stronger than an ex-ante bank portfolio. | **Bound the damage:** state the 10 of 12 full-window coverage evidence and use the results only as a continuous-coverage subset; CRSP/WRDS membership and delisting returns are required before a sector claim. |
| Fundamental availability dated at period end — **High** | If fundamentals had been used to select, rank, or rebalance, period-end dating would introduce look-ahead bias and could inflate alpha. | **Fix the input before using fundamentals:** use filing dates with the catalog's conservative 45-calendar-day availability policy. It does not affect this price-only return regression, but it limits any future fundamentals-based claim. |

Accordingly, these calculations can support a description of realized risk and benchmark-relative performance for the specified surviving tickers. They cannot support a claim about failed/delisted bank performance, an ex-ante bank-sector strategy, or a survivorship-free sector alpha.

## Exercise 3 — Step 9: Finding

The finding below is a data-backed observation from the week's calculations, not a new performance claim.

### Finding: the factor benchmark is not the same as either investable benchmark here

The equal-weight panel's annualized alpha is negative under all three regressions, but the estimated value changes materially with the benchmark: the French Mkt-RF and SPY estimates are near one another, while RSP produces a much less negative intercept and a higher R². The difference is economically meaningful relative to the point estimates, even though none of the alpha t-statistics is close to a conventional 5% threshold.

This makes the benchmark defence concrete: the return panel is equal-weighted and bank-heavy, while SPY is capitalization-weighted and Mkt-RF is a factor, not a tradable fund. The result does not establish that RSP is the right answer; it establishes that calling the panel's alpha a single unqualified number would overstate what this sample can say.

In [17]:
# Exercise 3 report and source log for Lab 3.
def md_table(frame, formatters):
    shown = frame.copy()
    for column, formatter in formatters.items():
        if column in shown:
            shown[column] = shown[column].map(formatter)
    shown.index.name = shown.index.name or "security"
    headers = [str(shown.index.name)] + [str(column) for column in shown.columns]
    rows = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    for index, row in shown.iterrows():
        rows.append("| " + " | ".join([str(index)] + [str(value) for value in row]) + " |")
    return "\n".join(rows)

step3_markdown = md_table(step3, {"CAGR": lambda x: f"{x:.2%}", "ann vol": lambda x: f"{x:.2%}", "Sharpe (excess, arithmetic)": lambda x: f"{x:.3f}", "CAGR/vol, rf=0": lambda x: f"{x:.3f}"})
step4_markdown = md_table(step4, {"α/yr": lambda x: f"{x:.2%}", "t(α)": lambda x: f"{x:.3f}", "β": lambda x: f"{x:.3f}", "t(β)": lambda x: f"{x:.3f}", "R²": lambda x: f"{x:.3f}", "resid vol": lambda x: f"{x:.2%}"})
step5_markdown = md_table(step5, {"β": lambda x: f"{x:.3f}", "total": lambda x: f"{x:.1%}", "systematic": lambda x: f"{x:.1%}", "idiosyncratic": lambda x: f"{x:.1%}", "share": lambda x: f"{x:.0%}"})
step6_markdown = md_table(step6_beta, {"beta": lambda x: f"{x:.3f}"})
step7_markdown = md_table(step7, {"alpha_per_year": lambda x: f"{x:.2%}", "t_alpha": lambda x: f"{x:.3f}", "beta": lambda x: f"{x:.3f}", "R_squared": lambda x: f"{x:.3f}"})

report = f"""# Exercise 3 — Lab 3 record

## Step 1 — conventions

Return type: simple monthly returns from adjusted month-end close (`yfinance`, `auto_adjust=True`).  
Frequency: monthly; annualization ×12 for means and alphas, ×√12 for volatilities.  
Risk-free: RF column, Ken French Data Library, Fama/French 3 Factors monthly.  
Window: 2015-07 to 2025-06, 120 months.  
Market series: French Mkt-RF (regression); SPY (investable benchmark).  
Equal-weight: rebalanced monthly to 1/N.  
Standard deviation: sample (`ddof=1`).

## Step 3 — Sharpe comparison

{step3_markdown}

Rank changes occur between BAC and IBM, among C, WFC, and ZION, and between KEY and TFC. I use the arithmetic-excess-return Sharpe ratio for the course because it matches the stated excess-return convention.

## Step 4 — Jensen regressions

{step4_markdown}

I would not stand behind any alpha as statistically reliable: none has an absolute alpha t-statistic near the usual 5% threshold of about 1.98. The equal-weight t-statistic reflects both shared market exposure and diversification of name-specific residual variation; it need not equal any member's t-statistic.

## Step 5 — risk decomposition

{step5_markdown}

The equal-weight panel's idiosyncratic volatility is {ew_idio:.2%}, versus a median single-name idiosyncratic volatility of {median_single_idio:.2%}. Equal-weighting 11 names reduced company-specific risk, but substantial residual risk remains because the panel is concentrated in related financial firms.

## Step 6 — independent JPM beta verification

Yahoo Finance's JPM quote page reported Beta (5Y Monthly) = {VENDOR_BETA:.2f}, retrieved {VENDOR_RETRIEVAL_DATE}.  

{step6_markdown}

I matched identity, frequency, 60 complete-month window, simple adjusted-close returns, SPY regressor, and an intercept OLS formula. Yahoo does not disclose all conventions, so I assume adjusted monthly total returns against an S&P 500 proxy; the {abs(matched_beta - VENDOR_BETA):.3f} difference is provisional and consistent with rounding.

## Step 7 — benchmark defence

{step7_markdown}

The strongest objection is that SPY's capitalization weighting is unlike the equal-weighted, bank-heavy panel; RSP is a plausible alternative and its alpha is materially different. I concede that benchmark sensitivity. Mkt-RF remains the stated factor benchmark and SPY the predeclared broad investable benchmark, but I report all three and make no benchmark-independent alpha claim.

## Step 8 — What these numbers can and cannot support

A **Blocker remains open**, so no headline alpha is reported for the full U.S. bank sector. These are survivor-subset descriptive results only.

| Open issue | Effect on alpha | Response |
|---|---|---|
| Requested names absent — **Blocker** | Missing failed/unavailable names leave a full-universe alpha unidentified. | **Restrict claim:** do not call this a survivorship-free bank-sector result. |
| End-of-sample universe — **High** | Survivor selection can bias alpha and historical performance. | **Bound damage:** report continuous-coverage subset only; obtain CRSP/WRDS membership and delisting returns for a sector claim. |
| Fundamental availability date — **High** | Period-end dating would create look-ahead bias if fundamentals drove a strategy. | **Fix input:** use filing date plus a 45-calendar-day availability policy before fundamental use. |

The panel can describe the realized surviving tickers under these conventions. It cannot support failed/delisted-name performance, an ex-ante bank-sector strategy, or a survivorship-free sector alpha.

## Step 9 — finding

The equal-weight alpha is negative under all three benchmarks, but RSP is materially less negative and has a higher R² than Mkt-RF or SPY. Since no alpha t-statistic is near conventional significance, this is evidence of benchmark sensitivity, not evidence that one benchmark establishes a reliable alpha.

## AI disclosure

Codex was used to help implement the calculations, assemble the requested markdown record, and explain the documented limitations. I reviewed the data, conventions, calculations, and written claims; the final submission retains the open Blocker rather than treating it as resolved.
"""
(LAB3_OUTPUT_DIR / "lab3_exercise3.md").write_text(report)
print("Wrote", LAB3_OUTPUT_DIR / "lab3_exercise3.md")

Wrote lab3_exercise3.md
